# Linear Regression 

In [2]:

# 1) Import
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)

In [3]:

# 2) Load train & test 
train_df = pd.read_csv("../data/processed/train_data_final.csv")
test_df = pd.read_csv("../data/processed/test_data_final.csv")

In [4]:
 
# 3) Split features/target in train, test
TARGET = "quantity_sold"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

has_y_test = TARGET in test_df.columns
if has_y_test:
    X_test = test_df.drop(columns=[TARGET])
    y_test = test_df[TARGET]
else:
    X_test = test_df.copy()
    y_test = None

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

X_train: (15848, 54), y_train: (15848,)


In [13]:
# 4) Feature selection
k_best = 40
selector = SelectKBest(score_func=f_regression, k=k_best)
selector.fit(X_train, y_train)
selected_features = X_train.columns[selector.get_support()].tolist()

print(f"Selected {len(selected_features)} features (KBest, k={k_best})")

X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

Selected 40 features (KBest, k=40)


In [14]:
# 5) Model tuning with Grid Search

# Define models and parameter grids
models_config = {
    'ridge': {
        'model': Ridge(),
        'params': {'model__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10, 50, 100, 200, 500, 1000]}
    },
    'lasso': {
        'model': Lasso(max_iter=5000, random_state=42),
        'params': {'model__alpha': [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50]}
    },
    'elasticnet': {
        'model': ElasticNet(max_iter=5000, random_state=42),
        'params': {'model__alpha':[0.0001, 0.001, 0.01, 0.1, 0.5, 1, 5, 10], 'model__l1_ratio': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]}
    }
}

results = {}
best_score = -np.inf
best_model = None
best_model_name = ""

# Train and tune each model
for name, config in models_config.items():
    print(f"\nTuning {name}...")
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', config['model'])
    ])
    
    grid_search = GridSearchCV(
        pipeline, config['params'], 
        cv=5, scoring='r2', n_jobs=-1, verbose=0
    )
    
    start_time = time.time()
    grid_search.fit(X_train_selected, y_train)
    training_time = time.time() - start_time
    
    results[name] = {
        'best_model': grid_search.best_estimator_,
        'best_params': grid_search.best_params_,
        'best_score': grid_search.best_score_,
        'cv_scores': grid_search.cv_results_['mean_test_score'],
        'training_time': training_time
    }
    
    print(f"  Best CV R2: {grid_search.best_score_:.4f}")
    print(f"  Best params: {grid_search.best_params_}")
    print(f"  Time: {training_time:.2f}s")
    
    # Track best model
    if grid_search.best_score_ > best_score:
        best_score = grid_search.best_score_
        best_model = grid_search.best_estimator_
        best_model_name = name



Tuning ridge...
  Best CV R2: 0.8964
  Best params: {'model__alpha': 50}
  Time: 5.59s

Tuning lasso...
  Best CV R2: 0.8964
  Best params: {'model__alpha': 0.0001}
  Time: 26.80s

Tuning elasticnet...
  Best CV R2: 0.8964
  Best params: {'model__alpha': 0.001, 'model__l1_ratio': 0.1}
  Time: 76.05s


In [15]:
print(f"✅ SELECTED BEST MODEL: {best_model_name.upper()}")

✅ SELECTED BEST MODEL: RIDGE


In [18]:
# 6) Evaluate on test set
if has_y_test:
    y_pred = best_model.predict(X_test_selected)
    
    metrics = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'MSE': mean_squared_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    }
    
    print("\nTest Set Performance:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    
    # Create results dataframe with ONLY 2 columns
    results_df = pd.DataFrame({
        'quantity_sold_ground_truth': y_test.values,
        'quantity_sold_predicted': y_pred
    }, index=test_df.index)


Test Set Performance:
MAE: 0.5290
MSE: 0.5320
RMSE: 0.7294
R2: 0.8983


In [19]:
# 7) Save results

predictions_file = f"lr_predictions.csv"
results_df.to_csv(predictions_file, index=True)